# *Stoic* — Protein Stoichiometry Prediction

**Fast and accurate protein stoichiometry prediction.**

Enter one protein sequence per unique chain/entity. *Stoic* predicts how many copies of each chain are present in the assembled complex.

---

In [ ]:
#@title **Setup** — install dependencies & load model { display-mode: "form" }
#@markdown *Run this cell once. It takes ~1 min on a T4 GPU.*

import importlib

if importlib.util.find_spec("stoic") is None:
    !pip install -q --no-deps "stoic @ git+https://github.com/PickyBinders/stoic.git"
    !pip install -q torch-geometric loguru bitsandbytes peft
    print("Packages installed.")
else:
    print("Packages already installed, skipping.")

import time
import traceback

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from stoic.model import Stoic

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if "model" not in dir() or not isinstance(model, Stoic):
    model = Stoic.from_pretrained("PickyBinders/stoic").to(device).eval()
    print("Model loaded.")
else:
    model = model.to(device)
    print("Model already loaded, skipping.")

In [ ]:
#@title **Predict stoichiometry** { display-mode: "form" }
import ipywidgets as widgets
from google.colab import files as colab_files

seq_boxes = []
output = widgets.Output()

def _chain_label(i):
    return chr(ord("A") + i)

def _make_seq_row(index, value=""):
    text = widgets.Textarea(
        value=value,
        placeholder="Paste protein sequence…",
        layout=widgets.Layout(width="95%", height="60px"),
    )
    label = widgets.HTML(f"<b>Chain {_chain_label(index)}</b>")
    return widgets.VBox([label, text])

def _rebuild_chain_labels():
    for i, box in enumerate(seq_boxes):
        box.children[0].value = f"<b>Chain {_chain_label(i)}</b>"

def add_chain(_=None):
    if len(seq_boxes) >= 26:
        return
    seq_boxes.append(_make_seq_row(len(seq_boxes)))
    _rebuild_chain_labels()
    chains_container.children = list(seq_boxes)

def remove_chain(_=None):
    if len(seq_boxes) <= 1:
        return
    seq_boxes.pop()
    _rebuild_chain_labels()
    chains_container.children = list(seq_boxes)

chains_container = widgets.VBox()

seq_boxes.append(_make_seq_row(0, "MKTLLILTLFLAIAASSASA"))
seq_boxes.append(_make_seq_row(1, "MGSSHHHHHHSSGLVPR"))
chains_container.children = list(seq_boxes)

add_btn = widgets.Button(description="+ Add chain", button_style="info",
                         layout=widgets.Layout(width="120px"))
remove_btn = widgets.Button(description="- Remove chain", button_style="warning",
                            layout=widgets.Layout(width="140px"))
add_btn.on_click(add_chain)
remove_btn.on_click(remove_chain)

top_n_slider = widgets.IntSlider(value=3, min=1, max=10, step=1,
                                 description="Top N:", style={"description_width": "60px"})
weights_cb = widgets.Checkbox(value=True, description="Return residue-level interface weights",
                              style={"description_width": "initial"})

predict_btn = widgets.Button(description="Predict Stoichiometry",
                             button_style="success",
                             layout=widgets.Layout(width="200px", height="36px"))

weights_tab = widgets.Tab(layout=widgets.Layout(width="100%"))
dl_stoich_btn = widgets.Button(description="Download stoichiometry.csv",
                               button_style="info", layout=widgets.Layout(width="220px"))
dl_weights_btn = widgets.Button(description="Download residue_weights.csv",
                                button_style="info", layout=widgets.Layout(width="240px"))

_last_stoich_df = [None]
_last_weights_df = [None]

def _dl_stoich(_=None):
    if _last_stoich_df[0] is not None:
        _last_stoich_df[0].to_csv("stoichiometry.csv", index=False)
        colab_files.download("stoichiometry.csv")

def _dl_weights(_=None):
    if _last_weights_df[0] is not None:
        _last_weights_df[0].to_csv("residue_weights.csv", index=False)
        colab_files.download("residue_weights.csv")

dl_stoich_btn.on_click(_dl_stoich)
dl_weights_btn.on_click(_dl_weights)

def _build_weight_tabs(weights_df, chain_labels):
    tab_children = []
    for i, label in enumerate(chain_labels):
        chain_name = f"Chain {label}"
        grp = weights_df[weights_df["Chain"] == chain_name]
        fig, ax = plt.subplots(figsize=(12, 3.5))
        ax.fill_between(grp["Position"], 0, grp["Weight"], alpha=0.3)
        ax.plot(grp["Position"], grp["Weight"], linewidth=1.2)
        ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
        ax.set_xlabel("Residue Position")
        ax.set_ylabel("Interface Weight")
        ax.set_title(f"Residue-level Interface Weights - {chain_name}")
        ax.set_xlim(grp["Position"].min(), grp["Position"].max())
        ax.set_ylim(0, 1)
        plt.tight_layout()
        tab_out = widgets.Output()
        with tab_out:
            plt.show()
        tab_children.append(tab_out)

    weights_tab.children = tab_children
    for i, label in enumerate(chain_labels):
        weights_tab.set_title(i, f"Chain {label}")

def on_predict(_=None):
    output.clear_output(wait=True)
    weights_tab.children = []
    dl_box.layout.display = "none"

    with output:
        try:
            sequences = [box.children[1].value.strip() for box in seq_boxes
                         if box.children[1].value.strip()]
            if not sequences:
                print("Please enter at least one sequence.")
                return
            if len(sequences) > 26:
                print("Maximum 26 unique chains supported.")
                return

            chain_labels = [_chain_label(i) for i in range(len(sequences))]
            top_n = top_n_slider.value
            return_weights = weights_cb.value

            print(f"Predicting stoichiometry for {len(sequences)} chain(s)...")
            for label, seq in zip(chain_labels, sequences):
                preview = seq[:50] + "..." if len(seq) > 50 else seq
                print(f"  Chain {label}: {preview}")

            start = time.time()
            with torch.no_grad():
                raw = model.predict_stoichiometry(
                    sequences, top_n=top_n, return_residue_weights=return_weights
                )
            elapsed = time.time() - start

            if return_weights:
                results, residue_predictions = raw
            else:
                results = raw

            # --- stoichiometry table ---
            stoich_rows = []
            for rank, candidate in enumerate(results, 1):
                copies = [candidate.get(seq, 0) for seq in sequences]
                stoich_rows.append({
                    "Rank": rank,
                    **{f"Chain {l}": c for l, c in zip(chain_labels, copies)},
                    "Score": candidate.get("rank", 0),
                    "Probability": candidate.get("probability", 0),
                })
            stoich_df = pd.DataFrame(stoich_rows)
            _last_stoich_df[0] = stoich_df

            header = "| Rank | " + " | ".join(f"Chain {l}" for l in chain_labels) + " | Stoichiometry | Score | Probability |"
            separator = "|------|" + "|".join("------" for _ in chain_labels) + "|---------------|-------|-------------|"
            md_rows = []
            for rank, candidate in enumerate(results, 1):
                copies = [candidate.get(seq, 0) for seq in sequences]
                stoich = "".join(f"{l}<sub>{c}</sub>" for l, c in zip(chain_labels, copies))
                score = candidate.get("rank", 0)
                prob = candidate.get("probability", 0)
                md_rows.append(f"| {rank} | " + " | ".join(str(c) for c in copies) + f" | {stoich} | {score:.2f} | {prob:.2e} |")

            table_md = "\n".join([header, separator] + md_rows)
            legend = "\n\n**Sequences:**\n"
            for label, seq in zip(chain_labels, sequences):
                preview = seq[:50] + "..." if len(seq) > 50 else seq
                legend += f"- **Chain {label}**: `{preview}`\n"

            display(Markdown(table_md + legend))
            print(f"\nRuntime: {elapsed:.2f}s")

            # --- residue weights ---
            if return_weights:
                pred_residues = residue_predictions["pred_residues"]
                attention_mask = residue_predictions["attention_mask"]
                seqs = residue_predictions["sequences"]

                records = []
                for i, seq in enumerate(seqs):
                    mask = ~(attention_mask[i].astype(bool))
                    weights = pred_residues[i][mask]
                    for pos, w in enumerate(weights, 1):
                        records.append({"Chain": f"Chain {chain_labels[i]}",
                                        "Position": pos, "Residue": seq[pos - 1],
                                        "Weight": float(w)})
                weights_df = pd.DataFrame(records)
                _last_weights_df[0] = weights_df
                dl_weights_btn.layout.display = ""
            else:
                _last_weights_df[0] = None
                dl_weights_btn.layout.display = "none"

            dl_box.layout.display = ""

        except Exception:
            traceback.print_exc()

    if return_weights and _last_weights_df[0] is not None:
        _build_weight_tabs(_last_weights_df[0], chain_labels)

predict_btn.on_click(on_predict)

dl_box = widgets.HBox([dl_stoich_btn, dl_weights_btn],
                       layout=widgets.Layout(display="none", margin="8px 0"))

display(widgets.HTML("<h3>Protein Sequences</h3><p>One sequence per box. Use the buttons to add/remove chains.</p>"))
display(chains_container)
display(widgets.HBox([add_btn, remove_btn]))
display(widgets.HTML("<hr><h3>Options</h3>"))
display(top_n_slider)
display(weights_cb)
display(widgets.HTML("<hr>"))
display(predict_btn)
display(output)
display(dl_box)
display(widgets.HTML("<h3>Residue-level Interface Weights</h3>"))
display(weights_tab)